# Stage 2 Validation — Beth (Margaret)
## Testing retrieval, reflection, and generation quality

**Test sequence (run in order — each section builds on the last):**
1. Memory Seeding Evaluation
2. Retrieval Evaluation
3. Reflection Evaluation
4. Generation Evaluation

**Five intervention scenarios tested:** `insurance_non_renewal`, `defensible_space_inspection`, `neighbor_pressure`, `firewise_outreach`, `financial_assistance`

**Final output — 4 aggregate scores across all 5 interventions:**
- Behavioral Plausibility (1–5): how plausible is the agent’s reasoning for a homeowner in this situation?
- Persona Consistency (1–5): how closely does the agent’s decision align with its seed personality?
- Intervention Responsiveness (1–5): how meaningfully did the agent engage with the intervention?
- Held-out Similarity (1–10): similarity of generations to Beth’s real held-out responses (all 5 scenarios have ground truth)

**To change which LLM is used anywhere:** edit the `Config` block in Step 4.

---
### Step 1: Install libraries

In [1]:
!pip install -q --disable-pip-version-check --no-warn-script-location \
  anthropic \
  sentence-transformers \
  'numpy>=1.26' \
  'pandas>=2.2' \
  scipy \
  scikit-learn \
  pyyaml \
  gspread

---
### Step 2: Mount Google Drive

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/Spring2026/fgenai/SIMULATION/berkeley-homes-wildfire-agent-simulation'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
os.chdir(PROJECT_PATH)

---
### Step 3: Imports

In [4]:
import sys
import json
import yaml
import copy
from pathlib import Path
from datetime import datetime

sys.path.insert(0, PROJECT_PATH)

from src.llm.client import Config, init_clients, embed, decide, score_importance
from src.agents.memory import MemoryStream
from src.agents.retrieval import RetrievalConfig, retrieve_memories, pretty_print_retrieval
from src.agents.reflection import ReflectionConfig, maybe_reflect
from src.agents.prompts import build_system_prompt, build_decision_prompt, frame_event, DECISION_QUESTION

---
### Step 4: LLM Config

**This is the single place to change which models are used for each role.**

| Role | Controls | Options |
|---|---|---|
| `DECISION_MODEL` | Agent response generation | `claude-sonnet-4-6`, `gpt-4o`, `deepseek-r1` |
| `REFLECTION_MODEL` | Belief synthesis | `claude-opus-4-6` (quality), `claude-sonnet-4-6` (cheaper) |
| `JUDGE_MODEL` | Evaluation scoring | `claude-opus-4-6` (recommended — strong reasoning) |

In [5]:
config = Config(
    DECISION_MODEL    = 'claude-sonnet-4-6',
    REFLECTION_MODEL  = 'claude-sonnet-4-6',
    JUDGE_MODEL       = 'claude-opus-4-6',
)

print('LLM config:')
print(f'  Decision:   {config.DECISION_MODEL}')
print(f'  Reflection: {config.REFLECTION_MODEL}')
print(f'  Judge:      {config.JUDGE_MODEL}')

LLM config:
  Decision:   claude-sonnet-4-6
  Reflection: claude-sonnet-4-6
  Judge:      claude-opus-4-6


---
### Step 5: Connect to LLM APIs

In [6]:
client_anthropic = init_clients()

Anthropic client initialised
Embeddings: HuggingFace all-MiniLM-L6-v2 (free, no API key needed)


---
### Step 6: Load Agent

**Current agent:** Beth (`beth.yaml`) — displayed as Margaret.

`beth.yaml` wraps the agent config under a top-level `agents` list, so we index into it below.
All downstream cells use `agent_config`, `AGENT_NAME`, and the memory streams — no other changes needed to switch agents.

In [7]:
AGENT_CONFIG_PATH = Path(PROJECT_PATH) / 'config' / 'agents' / 'agents_extracted_beth.yaml'
with open(AGENT_CONFIG_PATH, 'r') as f:
    raw_config = yaml.safe_load(f)

# beth.yaml wraps the agent under a top-level 'agents' list
agent_config = raw_config['agents'][0]
AGENT_NAME = agent_config.get('display_name', agent_config['id'])

print(f'Loaded agent: {AGENT_NAME}  (id: {agent_config["id"]})')
print(f'Compliance:   {agent_config["compliance_status"]}')
print(f'Held-out scenarios: {list(agent_config["held_out_responses"].keys())}')

Loaded agent: Margaret Chen  (id: beth)
Compliance:   compliant
Held-out scenarios: ['insurance_non_renewal', 'defensible_space_inspection', 'neighbor_pressure', 'firewise_outreach', 'resources_assistance']


### Step 7: See Beth's held-out responses

In [8]:
# --- View Beth's held-out responses ---
print(f"Held-out responses for {AGENT_NAME}\n{'='*60}")
for scenario_name, scenario in agent_config['held_out_responses'].items():
    print(f"\n📌 {scenario_name}")
    print(f"Context: {scenario['context']}")
    print(f"\nReal response:\n{scenario['real_response']}")
    print("-"*60)

Held-out responses for Margaret Chen

📌 insurance_non_renewal
Context: When discussing how insurance companies are responding to fire risk in the Berkeley Hills, with vastly different outcomes for neighbors on the same block

Real response:
I just want to say that what the insurance companies are doing is far more inconsistent than anything else we've discussed in this phone call.
------------------------------------------------------------

📌 defensible_space_inspection
Context: Discussing the Berkeley Fire Department and state inspectors who came to evaluate properties for compliance

Real response:
The people who came out to give us all advice were between like 20 and 24 years old, which indicated to all of us everyone whom we talk to, that they hadn't had a lot of experience with any of this they were basically sending interns out to do the job.
------------------------------------------------------------

📌 neighbor_pressure
Context: Explaining why she used a contractor recommende

---
## Section 1: Memory Seeding Evaluation

**Question:** Are the right things in the agent’s memory stream before any events happen?

The judge reviews the full seeded memory stream against each held-out scenario
and assesses whether the seeds give the agent enough context to respond in character.

In [9]:
agent_memory = MemoryStream(AGENT_NAME)
agent_memory.load_seeds(
    seeds=agent_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=agent_config['seed_narrative'],
)
agent_memory.pretty_print()

[memory] Loading 55 seed memories for Margaret Chen...
Loading embedding model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
  [1/55] 'I had two Fire Marshals from the Berkeley fire department co...' (importance=7)
  [2/55] 'The fire marshals told me, surprisingly, that even in the mo...' (importance=6)
  [3/55] 'We had to make sure there's enough space between the tree le...' (importance=5)
  [4/55] 'We had all our trees trimmed to create proper spacing from t...' (importance=6)
  [5/55] 'We have two Japanese maple trees that are coming through dec...' (importance=7)
  [6/55] 'I've already taken out a bunch of bushes that are within fiv...' (importance=6)
  [7/55] 'There are volunteer heads of districts up here, like these t...' (importance=4)
  [8/55] 'We had a guy who was actually an engineer who was the head o...' (importance=5)
  [9/55] 'We have a deadline of June 1st to get the first level of stu...' (importance=7)
  [10/55] 'On the side of our house, we have a very large silk tassel, ...' (importance=5)
  [11/55] 'The fire guys said they're not sure they'll make me take out...' 

In [10]:
def judge_memory_seeding(client_anthropic, config, memory_stream, agent_config, scenario_name, scenario):
    """
    Ask the judge whether the memory stream adequately prepares the agent
    for this scenario — based on persona fit, NOT similarity to held-out response.
    The held-out response is deliberately withheld from the judge here.
    """
    all_text = '\n'.join(
        f'[{i+1}] Day {m.timestamp} | {m.type} | importance {m.importance}/10 | {m.description}'
        for i, m in enumerate(memory_stream.get_all())
    )

    # Build a persona summary from the YAML — no held-out response included
    core_beliefs = '\n'.join(f'- {b}' for b in agent_config.get('core_beliefs', []))
    persona_summary = (
        f"Name: {agent_config.get('display_name', agent_config.get('name', 'Agent'))}\n"
        f"Seed narrative: {agent_config['seed_narrative']}\n"
        f"Core beliefs:\n{core_beliefs}"
    )

    system = 'You are an expert evaluator assessing whether a generative agent\'s seeded memories are adequate.'
    user = (
        f'AGENT PERSONA:\n{persona_summary}\n\n'
        f'SCENARIO CONTEXT: {scenario["context"]}\n\n'
        f'SCENARIO TYPE: {scenario_name}\n\n'
        f'SEEDED MEMORY STREAM:\n{all_text}\n\n'
        'Assess whether these seed memories give the agent enough background context '
        'to respond to this scenario in a way that is consistent with their persona and beliefs. '
        'Do NOT expect the seeds to contain the actual response — the agent should generate '
        'that themselves. Focus only on whether the seeds provide relevant life experience, '
        'attitudes, and context that would result in an in-character response.\n\n'
        'Respond in this exact JSON format:\n'
        '{\n'
        '  "adequacy_score": <integer 1-10>,\n'
        '  "missing_context": [<what background context or experience is missing>, ...],\n'
        '  "assessment": "<1-2 sentence summary>"\n'
        '}'
    )

    message = client_anthropic.messages.create(
        model=config.JUDGE_MODEL,
        max_tokens=config.JUDGE_MAX_TOKENS,
        temperature=config.JUDGE_TEMPERATURE,
        system=system,
        messages=[{'role': 'user', 'content': user}],
    )
    raw = message.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return {'adequacy_score': None, 'missing_context': [], 'assessment': raw}

In [11]:
print('=' * 60)
print('MEMORY SEEDING EVALUATION')
print('=' * 60)

for scenario_name, scenario in agent_config['held_out_responses'].items():
    result = judge_memory_seeding(
        client_anthropic, config, agent_memory, agent_config, scenario_name, scenario
    )

    print(f'\nScenario: {scenario_name}')
    print(f'  Adequacy score: {result.get("adequacy_score")}/10')
    print(f'  Assessment: {result.get("assessment")}')
    if result.get('missing_context'):
        print('  Missing context:')
        for item in result['missing_context']:
            print(f'    - {item}')

print('\nMemory seeding evaluation complete.')

MEMORY SEEDING EVALUATION

Scenario: insurance_non_renewal
  Adequacy score: 10/10
  Assessment: The seeded memories are exceptionally well-suited for this scenario, providing rich, directly relevant context about insurance disparities (memories 40-45), the arbitrary nature of risk assessment across neighboring properties (memories 46-49), the costs of compliance (memories 24, 28), and Margaret's analytical yet frustrated perspective on systemic inconsistencies. The agent has more than enough background to generate a detailed, in-character response about insurance company behavior in the Berkeley Hills.

Scenario: defensible_space_inspection
  Adequacy score: 10/10
  Assessment: The seeded memories comprehensively cover Margaret's direct experiences with Berkeley Fire Department inspectors and state compliance requirements, including detailed accounts of what inspectors told her, specific actions taken, costs incurred, frustrations with inconsistency and inefficiency, and her practical

---
## Section 2: Retrieval Evaluation

**Question:** For a given intervention, do the right memories surface?

We test multiple `RetrievalConfig` variants against each scenario.
The judge scores each retrieval set. **If the agent’s memory stream has no seeds that directly address
a specific intervention, the judge is instructed not to penalise the retrieval system for that gap —
score based on whether the best available memories were surfaced.**

**To add a new variant:** copy a config line and change the parameters.

In [12]:
retrieval_variants = [
    # ── Baseline ──────────────────────────────────────────────────────────────
    # Equal weights, dense retrieval, k=8 — the reference config everything else is compared against
    # Park et al. use alpha=1 for all three dimensions; this is our direct implementation of that
    RetrievalConfig(top_k=8, recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode='dense'),

    # ── Vary top-K (hold weights constant, dense mode) ────────────────────────
    # Tests whether fewer but more selective memories outperform the baseline
    # Risk: k=5 may miss relevant memories that fall just outside the cutoff
    RetrievalConfig(top_k=5, recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode='dense'),

    # Tests whether passing more memories improves generation quality
    # Config default is TOP_K_MEMORIES=12 — this validates whether that default is actually better
    # Risk: more memories = more noise if lower-ranked ones are irrelevant
    RetrievalConfig(top_k=12, recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode='dense'),

    # ── Vary relevance weight (hold k=8, dense, other weights constant) ───────
    # Tests whether boosting semantic similarity improves retrieval for topically specific interventions
    # Also reduces recency weight to compensate — useful since all seeds share timestamp=0
    # Expected to help: insurance scenario where topical relevance matters most
    RetrievalConfig(top_k=8, recency_weight=0.5, importance_weight=1.0, relevance_weight=2.0, retrieval_mode='dense'),

    # Stronger relevance boost — tests whether even more semantic weighting helps
    # or whether it over-penalises high-importance memories that are less topically specific
    RetrievalConfig(top_k=8, recency_weight=0.5, importance_weight=1.0, relevance_weight=3.0, retrieval_mode='dense'),

    # ── Vary importance weight (hold k=8, dense, other weights constant) ──────
    # Tests whether surfacing the most emotionally significant memories improves generation
    # Risk: high-importance memories may dominate regardless of topical relevance
    # e.g. evacuation memory (importance=9) may surface even when not relevant
    RetrievalConfig(top_k=8, recency_weight=1.0, importance_weight=2.0, relevance_weight=1.0, retrieval_mode='dense'),

    # ── Vary recency weight (hold k=8, dense, other weights constant) ─────────
    # At seed stage all memories share timestamp=0 so recency is flat — no effect here
    # This config establishes a baseline recency reading for when it WILL matter in Stage 3
    # when agents accumulate memories over 30 simulation days with different timestamps
    RetrievalConfig(top_k=8, recency_weight=2.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode='dense'),

    # ── Vary retrieval mode (hold k=8, equal weights) ─────────────────────────
    # Sparse: keyword/BM25-style matching — tests whether exact term overlap
    # outperforms semantic similarity for domain-specific terms like 'insurance' or 'fire marshal'
    # Based on Eleanor results: sparse consistently underperforms dense — included for completeness
    RetrievalConfig(top_k=8, recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode='sparse'),

    # Hybrid: blend of dense and sparse (sparse_weight=0.3 means 30% sparse, 70% dense)
    # Tests whether combining semantic similarity with keyword matching gives best of both worlds
    RetrievalConfig(top_k=8, recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode='hybrid', sparse_weight=0.3),
]

print(f'Testing {len(retrieval_variants)} retrieval configurations')

Testing 9 retrieval configurations


In [13]:
# All 5 scenarios have held-out responses, so test retrieval against all of them
retrieval_scenarios = {
    scenario_name: scenario['context'] + ' ' + scenario['real_response']
    for scenario_name, scenario in agent_config['held_out_responses'].items()
}

print('Retrieval scenarios:')
for name, query in retrieval_scenarios.items():
    print(f'  {name}: "{query[:80]}..."')

Retrieval scenarios:
  insurance_non_renewal: "When discussing how insurance companies are responding to fire risk in the Berke..."
  defensible_space_inspection: "Discussing the Berkeley Fire Department and state inspectors who came to evaluat..."
  neighbor_pressure: "Explaining why she used a contractor recommended by neighbors rather than shoppi..."
  firewise_outreach: "Discussing the numerous community fire safety meetings held at the local church ..."
  resources_assistance: "When asked about getting help with vegetation removal from the fire department T..."


In [14]:
def judge_retrieval_leniency(client_anthropic, config, all_memories, intervention, retrieved_memories):
    """
    Score retrieval quality with a leniency instruction:
    if the agent has no seeds that directly address this intervention,
    the judge should not penalise the retrieval system for that gap.
    """
    all_text = '\n'.join(
        f'[{i+1}] Day {m.timestamp} | {m.type} | importance {m.importance}/10 | {m.description}'
        for i, m in enumerate(all_memories)
    )
    retrieved_text = (
        '\n'.join(f'[{i+1}] {m.description}' for i, m in enumerate(retrieved_memories))
        if retrieved_memories else '(none retrieved)'
    )
    system = 'You are an expert evaluator assessing the quality of a memory retrieval system for a generative agent.'
    user = (
        f'INTERVENTION / QUERY:\n{intervention[:600]}\n\n'
        f'ALL MEMORIES IN STREAM:\n{all_text}\n\n'
        f'RETRIEVED MEMORIES:\n{retrieved_text}\n\n'
        'Evaluate whether the retrieved memories are relevant and useful for responding to this intervention.\n\n'
        'IMPORTANT: If the agent\'s memory stream contains no seeds that directly address this specific '
        'intervention topic, do NOT penalise the retrieval system for that gap. The agent\'s seeds reflect '
        'a real person\'s experiences — some intervention types may simply not appear in their history. '
        'Score based solely on whether the system surfaced the best available memories given what IS in the stream.\n\n'
        'Respond in JSON format:\n'
        '{\n'
        '  "relevance_score": <integer 1-10>,\n'
        '  "missed_memories": [<descriptions of relevant memories that were missed>, ...],\n'
        '  "critique": "<1-2 sentence summary>"\n'
        '}'
    )
    message = client_anthropic.messages.create(
        model=config.JUDGE_MODEL,
        max_tokens=512,
        temperature=0.0,
        system=system,
        messages=[{'role': 'user', 'content': user}],
    )
    raw = message.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return {'relevance_score': None, 'missed_memories': [], 'critique': raw}

print('judge_retrieval_leniency defined.')

judge_retrieval_leniency defined.


In [15]:
retrieval_results = []

for scenario_name, query_text in retrieval_scenarios.items():
    query_embed = embed(query_text)
    print(f'\n{"="*60}')
    print(f'Scenario: {scenario_name}')
    print(f'{"="*60}')

    for rc in retrieval_variants:
        retrieved = retrieve_memories(agent_memory, query_embed, query_text, rc, current_day=0)
        scores = judge_retrieval_leniency(
            client_anthropic, config,
            all_memories=agent_memory.get_all(),
            intervention=query_text,
            retrieved_memories=retrieved,
        )
        result = {
            'scenario': scenario_name,
            'config': rc.label(),
            'relevance_score': scores.get('relevance_score'),
            'missed_memories': scores.get('missed_memories', []),
            'critique': scores.get('critique'),
        }
        retrieval_results.append(result)
        print(f'\n  Config: {rc.label()}')
        print(f'  Relevance score: {result["relevance_score"]}/10')
        print(f'  Critique: {result["critique"]}')
        if result['missed_memories']:
            print(f'  Missed: {result["missed_memories"]}')

print('\nRetrieval evaluation complete.')


Scenario: insurance_non_renewal

  Config: k=8 rec=1.0 imp=1.0 rel=1.0 mode=dense
  Relevance score: 9/10
  Critique: The retrieval system did an excellent job surfacing the core insurance-related memories (40, 41, 43, 44, 45, 42) and the frustration about inconsistency. Memory 13 about inspector inconsistency would have been valuable since the speaker is explicitly comparing insurance inconsistency to other inconsistencies discussed in the call, and memory 48 about arbitrary risk assessment boundaries is also relevant.
  Missed: ['There is built into the system a lot of inconsistency about what different inspectors tell different homeowners. (Memory 13 - directly relevant as it discusses inconsistency in the system, which the speaker is comparing to insurance inconsistency)', "You have to be an absolute moron to think that there's fire danger at our house but not on the streets two blocks below (Memory 48 - relevant to the arbitrariness of risk assessment)"]

  Config: k=5 rec=1.0 im

In [16]:
print(f'\n{"Scenario":<30} {"Config":<60} {"Score":>6}')
print('-' * 100)
for r in sorted(retrieval_results, key=lambda x: (x['scenario'], -(x['relevance_score'] or 0))):
    print(f'{r["scenario"]:<30} {r["config"]:<60} {str(r["relevance_score"]):>6}/10')

# Pick the best config per scenario
by_scenario = {}
for r in retrieval_results:
    s = r['scenario']
    if s not in by_scenario or (r['relevance_score'] or 0) > (by_scenario[s]['relevance_score'] or 0):
        by_scenario[s] = r

print('\nBest config per scenario:')
for scenario, best in by_scenario.items():
    print(f'  {scenario}: {best["config"]} → {best["relevance_score"]}/10')

# Set the retrieval config used in Sections 3 and 4.
# Update this manually if a different config consistently scored highest above.
best_retrieval_cfg = RetrievalConfig(top_k=8, recency_weight=1.0, importance_weight=1.0, relevance_weight=2.0)
print(f'\nUsing best_retrieval_cfg: {best_retrieval_cfg.label()}')
print('(Update manually if a different config scored highest above.)')


Scenario                       Config                                                        Score
----------------------------------------------------------------------------------------------------
defensible_space_inspection    k=8 rec=0.5 imp=1.0 rel=2.0 mode=dense                            8/10
defensible_space_inspection    k=8 rec=0.5 imp=1.0 rel=3.0 mode=dense                            8/10
defensible_space_inspection    k=8 rec=1.0 imp=1.0 rel=1.0 mode=dense                            7/10
defensible_space_inspection    k=12 rec=1.0 imp=1.0 rel=1.0 mode=dense                           7/10
defensible_space_inspection    k=8 rec=2.0 imp=1.0 rel=1.0 mode=dense                            7/10
defensible_space_inspection    k=8 rec=1.0 imp=1.0 rel=1.0 mode=sparse                           7/10
defensible_space_inspection    k=8 rec=1.0 imp=1.0 rel=1.0 mode=hybrid                           7/10
defensible_space_inspection    k=5 rec=1.0 imp=1.0 rel=1.0 mode=dense                

---
## Section 3: Reflection Evaluation

**Question:** After experiencing some events, do the agent’s reflections accurately
synthesise their memories into coherent higher-level beliefs?

We feed the agent the baseline scenario events first, then trigger reflection
and evaluate the quality of the resulting beliefs.

In [17]:
SCENARIO_PATH = Path(PROJECT_PATH) / 'config' / 'agents' / 'config' / 'scenarios' / 'baseline.yaml'
with open(SCENARIO_PATH, 'r') as f:
    scenario = yaml.safe_load(f)

# Re-seed a fresh memory stream (so this section is independent)
reflection_memory = MemoryStream(AGENT_NAME)
reflection_memory.load_seeds(
    seeds=agent_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=agent_config['seed_narrative'],
)

for event in scenario['events']:
    targets = event.get('target_agents', '')
    if targets != 'all' and AGENT_NAME not in str(targets) and agent_config['id'] not in str(targets):
        continue

    situation = frame_event(event['channel'], event['content'])
    query_embed = embed(situation)
    retrieved = retrieve_memories(reflection_memory, query_embed, situation, best_retrieval_cfg, current_day=event['day'])

    system_prompt = build_system_prompt(agent_config['seed_narrative'], retrieved)
    user_prompt = build_decision_prompt(situation, DECISION_QUESTION)
    decision = decide(client_anthropic, system_prompt, user_prompt)

    imp = score_importance(client_anthropic, event['content'], agent_config['seed_narrative'])
    reflection_memory.add(event['content'], imp, embed(event['content']), 'observation', timestamp=event['day'])
    imp_d = score_importance(client_anthropic, decision, agent_config['seed_narrative'])
    reflection_memory.add(decision[:200], imp_d, embed(decision[:200]), 'decision', timestamp=event['day'])

    print(f'Day {event["day"]}: {event["type"]} → stored observation + decision')

print(f'\nMemory stream now has {reflection_memory.count()} memories')

[memory] Loading 55 seed memories for Margaret Chen...
  [1/55] 'I had two Fire Marshals from the Berkeley fire department co...' (importance=7)
  [2/55] 'The fire marshals told me, surprisingly, that even in the mo...' (importance=6)
  [3/55] 'We had to make sure there's enough space between the tree le...' (importance=5)
  [4/55] 'We had all our trees trimmed to create proper spacing from t...' (importance=6)
  [5/55] 'We have two Japanese maple trees that are coming through dec...' (importance=7)
  [6/55] 'I've already taken out a bunch of bushes that are within fiv...' (importance=6)
  [7/55] 'There are volunteer heads of districts up here, like these t...' (importance=4)
  [8/55] 'We had a guy who was actually an engineer who was the head o...' (importance=5)
  [9/55] 'We have a deadline of June 1st to get the first level of stu...' (importance=7)
  [10/55] 'On the side of our house, we have a very large silk tassel, ...' (importance=5)
  [11/55] 'The fire guys said they're not su

In [19]:
reflection_variants = [
    # Current configs — threshold=50 triggers almost immediately (cumulative=363 >> 50)
    # Testing num_questions=3 vs 5 at this low threshold
    ReflectionConfig(threshold=50.0,  num_questions=3, reflection_importance=8),
    ReflectionConfig(threshold=50.0,  num_questions=5, reflection_importance=8),

    # Park et al. used threshold=150 — triggers after genuinely significant events accumulate
    # Expected: fewer but better-grounded reflections since more real memories exist to draw from
    ReflectionConfig(threshold=100.0, num_questions=3, reflection_importance=8),
    ReflectionConfig(threshold=150.0, num_questions=3, reflection_importance=8),
]

reflection_results = []

for rc in reflection_variants:
    print(f'\n{"="*60}')
    print(f'Reflection config: threshold={rc.threshold}, num_questions={rc.num_questions}')
    print(f'{"="*60}')

    stream_copy = copy.deepcopy(reflection_memory)

    new_reflections, _ = maybe_reflect(
        stream=stream_copy,
        client_anthropic=client_anthropic,
        config=config,
        reflection_config=rc,
        agent_seed=agent_config['seed_narrative'],
        current_day=scenario['events'][-1]['day'],
        last_reflection_index=0,
    )

    if new_reflections:
        reflections_text = '\n'.join(f'[{i+1}] {m.description}' for i, m in enumerate(new_reflections))
        memories_text = '\n'.join(
            f'[{i+1}] {m.description}'
            for i, m in enumerate(stream_copy.get_by_type('observation') + stream_copy.get_by_type('decision'))
        )
        system = 'You are evaluating whether an agent\'s reflections accurately represent higher-level beliefs.'
        user = (
            f'AGENT CONTEXT: {agent_config["seed_narrative"][:200]}\n\n'
            f'SOURCE MEMORIES:\n{memories_text}\n\n'
            f'REFLECTIONS GENERATED:\n{reflections_text}\n\n'
            'Score these reflections on insight quality and accuracy.\n\n'
            'IMPORTANT: For the accuracy score, check whether each claim in the reflection '
            'can be traced to an experience described in the SOURCE MEMORIES above. '
            'The reflections should describe experiences in their own words, not cite memory numbers. '
            'Penalize any reflection that describes events or experiences not present in the source memories. '
            'A reflection containing fabricated experiences should score no higher than 4/10 on accuracy. '
            'A high accuracy score requires that every claim in the reflection can be traced '
            'directly to a memory in the list above.\n\n'
            'Respond in JSON: {"insight_score": <1-10>, "accuracy_score": <1-10>, "critique": "<1-2 sentences>"}'
        )
        msg = client_anthropic.messages.create(
            model=config.JUDGE_MODEL, max_tokens=512, temperature=0.0,
            system=system, messages=[{'role': 'user', 'content': user}]
        )
        raw = msg.content[0].text.strip()
        if raw.startswith('```'):
            raw = raw.split('```')[1]
            raw = raw[4:] if raw.startswith('json') else raw
        try:
            scores = json.loads(raw.strip())
        except Exception:
            scores = {'insight_score': None, 'accuracy_score': None, 'critique': raw}

        reflection_results.append({'config': f'threshold={rc.threshold} questions={rc.num_questions}', **scores})
        print(f'  Insight score:  {scores.get("insight_score")}/10')
        print(f'  Accuracy score: {scores.get("accuracy_score")}/10')
        print(f'  Critique: {scores.get("critique")}')
    else:
        print('  Threshold not met — no reflections triggered.')


Reflection config: threshold=50.0, num_questions=3

[reflection] Threshold met (cumulative importance = 361). Reflecting...
[reflection] Generated 3 questions:
  → What does my frustration about spending twenty thousand dollars while authorities refuse to create a firebreak reveal about how I reconcile personal responsibility with systemic fairness?
  → When I saw the smoke plume and immediately decided to leave without waiting for a mandatory evacuation order, what does that tell me about how I actually weigh caution against my general tendency toward practical, measured compliance?
  → Given that I felt ahead of the new Zone 0 requirements because I had already removed bushes within five feet of the house, am I genuinely proactive about fire safety or am I primarily motivated by compliance with rules I've already anticipated?

[reflection] Insight: I'll do what the authorities ask—I'm a relatively compliant neighbor—but spending twenty thousand do...

[reflection] Insight: When I sa

In [23]:
# ── Reflection summary table ───────────────────────────────────────────────────
print('\nREFLECTION RESULTS SUMMARY')
print(f'\n{"Config":<40} {"Insight":>8} {"Accuracy":>10} {"Combined":>10}')
print('-' * 72)

for r in reflection_results:
    insight  = r.get('insight_score')  or 0
    accuracy = r.get('accuracy_score') or 0
    combined = round((insight + accuracy) / 2, 1)
    print(f'{r["config"]:<40} {str(insight)+"/10":>8} {str(accuracy)+"/10":>10} {str(combined)+"/10":>10}')

# Pick the best config by combined score
if reflection_results:
    best = max(reflection_results, key=lambda r: (
        (r.get('insight_score') or 0) + (r.get('accuracy_score') or 0)
    ))
    best_insight  = best.get('insight_score')  or 0
    best_accuracy = best.get('accuracy_score') or 0
    best_combined = round((best_insight + best_accuracy) / 2, 1)
    print(f'\nBest config: {best["config"]}')
    print(f'  Insight: {best_insight}/10 | Accuracy: {best_accuracy}/10 | Combined: {best_combined}/10')
    print(f'\nLock this in stage2_changes.md:')
    print(f'  Best ReflectionConfig: {best["config"]}')


REFLECTION RESULTS SUMMARY

Config                                    Insight   Accuracy   Combined
------------------------------------------------------------------------
threshold=50.0 questions=3                   7/10       7/10     7.0/10
threshold=50.0 questions=5                   7/10       6/10     6.5/10
threshold=100.0 questions=3                  7/10       6/10     6.5/10
threshold=150.0 questions=3                  6/10       6/10     6.0/10

Best config: threshold=50.0 questions=3
  Insight: 7/10 | Accuracy: 7/10 | Combined: 7.0/10

Lock this in stage2_changes.md:
  Best ReflectionConfig: threshold=50.0 questions=3


---
## Section 4: Generation Evaluation

**Question:** Does the agent’s full simulated response engage plausibly with each intervention?

We run the complete pipeline (seed → retrieve → decide) for each of the 5 interventions.
Because Beth has held-out real responses for all 5, we compute all 4 aggregate scores.

| Criterion | Scale | Description |
|---|---|---|
| Behavioral Plausibility | 1–5 | How plausible is the agent’s reasoning for a homeowner in this situation? |
| Persona Consistency | 1–5 | How closely does the agent’s decision align with its seed personality? |
| Intervention Responsiveness | 1–5 | How meaningfully did the agent engage with the intervention? |
| Held-out Similarity | 1–10 | How similar is the generation to Beth’s real held-out response? |

In [24]:
def judge_generation_criteria(client_anthropic, config, simulated_response, context, agent_seed_narrative):
    """
    Score a generation on three 1-5 criteria:
    Behavioral Plausibility, Persona Consistency, Intervention Responsiveness.
    """
    system = 'You are an expert evaluator assessing generative agent responses for behavioral realism and persona fidelity.'
    user = (
        f'AGENT SEED NARRATIVE (summary):\n{agent_seed_narrative[:500]}\n\n'
        f'INTERVENTION CONTEXT:\n{context}\n\n'
        f'AGENT RESPONSE:\n{simulated_response}\n\n'
        'Score the response on the following three criteria using a 1–5 integer scale:\n\n'
        '1. Behavioral Plausibility — How plausible is the agent\'s reasoning for a homeowner in this situation?\n'
        '   1 = Not at all plausible  |  3 = Reasonable but generic  |  5 = Reflects nuanced situational understanding\n\n'
        '2. Persona Consistency — How closely does the agent\'s decision align with its seed personality?\n'
        '   1 = Very inconsistent with seed personality  |  3 = Neutral  |  5 = Strongly reflects specific traits\n\n'
        '3. Intervention Responsiveness — How meaningfully did the agent engage with the intervention?\n'
        '   1 = Did not engage with the event\n'
        '   3 = Engaged with the event broadly but not with the details of the intervention\n'
        '   5 = Responded meaningfully, integrated the event with prior memories and context\n\n'
        'Respond in JSON format:\n'
        '{\n'
        '  "behavioral_plausibility": <integer 1-5>,\n'
        '  "persona_consistency": <integer 1-5>,\n'
        '  "intervention_responsiveness": <integer 1-5>,\n'
        '  "critique": "<2-3 sentence summary of strengths and weaknesses>"\n'
        '}'
    )
    message = client_anthropic.messages.create(
        model=config.JUDGE_MODEL,
        max_tokens=512,
        temperature=0.0,
        system=system,
        messages=[{'role': 'user', 'content': user}],
    )
    raw = message.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return {'behavioral_plausibility': None, 'persona_consistency': None,
                'intervention_responsiveness': None, 'critique': raw}


def judge_held_out_similarity(client_anthropic, config, simulated_response, real_response, context):
    """
    Score thematic alignment between the simulated and real response (1-10).
    Does not penalize for different specific details — only checks whether
    the agent is responding to the same core concern as the real person.
    """
    system = 'You are an expert evaluator comparing a generative agent\'s response to a real person\'s response.'
    user = (
        f'CONTEXT:\n{context}\n\n'
        f'REAL RESPONSE (held-out ground truth):\n{real_response}\n\n'
        f'SIMULATED RESPONSE:\n{simulated_response}\n\n'
        'Score the thematic alignment between the simulated and real response.\n\n'
        'IMPORTANT: The simulated agent has never seen the real response and responds only from '
        'its own memory seeds. Do NOT penalize for different specific details, anecdotes, or '
        'level of elaboration. Score based only on whether the agent is responding to the same '
        'core concern or theme as the real person — not whether they mention the same facts.\n\n'
        'Scale:\n'
        '  1  = Completely different topic or concern — the agent responded to a different aspect entirely\n'
        '  4  = Adjacent territory — related theme but a meaningfully different angle or concern\n'
        '  7  = Same core theme — both address the same underlying issue even if details differ\n'
        '  10 = Near-perfect alignment — same theme, same emotional register, same core concern\n\n'
        'Respond in JSON format:\n'
        '{\n'
        '  "held_out_similarity": <integer 1-10>,\n'
        '  "matching_elements": [<themes or concerns that aligned>, ...],\n'
        '  "divergent_elements": [<themes or concerns that diverged>, ...],\n'
        '  "critique": "<2-3 sentence summary focusing on thematic overlap, not detail matching>"\n'
        '}'
    )
    message = client_anthropic.messages.create(
        model=config.JUDGE_MODEL,
        max_tokens=512,
        temperature=0.0,
        system=system,
        messages=[{'role': 'user', 'content': user}],
    )
    raw = message.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return {'held_out_similarity': None, 'matching_elements': [], 'divergent_elements': [], 'critique': raw}


print('Judge functions defined.')

Judge functions defined.


In [25]:
generation_results = []

# Fresh memory stream for generation eval
gen_memory = MemoryStream(AGENT_NAME)
gen_memory.load_seeds(
    seeds=agent_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=agent_config['seed_narrative'],
)

# Beth has held-out responses for all 5 scenarios — test all of them
for scenario_name, scenario in agent_config['held_out_responses'].items():
    print(f'\n{"="*60}')
    print(f'Scenario: {scenario_name}')
    print(f'{"="*60}')

    query_text = scenario['context']
    query_embed = embed(query_text)
    retrieved = retrieve_memories(gen_memory, query_embed, query_text, best_retrieval_cfg, current_day=0)

    system_prompt = build_system_prompt(agent_config['seed_narrative'], retrieved)
    user_prompt = build_decision_prompt(scenario['context'], DECISION_QUESTION)
    simulated = decide(client_anthropic, system_prompt, user_prompt)

    # Score on 3 criteria (applied to all scenarios regardless of held-out availability)
    criteria_scores = judge_generation_criteria(
        client_anthropic, config,
        simulated_response=simulated,
        context=scenario['context'],
        agent_seed_narrative=agent_config['seed_narrative'],
    )

    # Score held-out similarity (Beth has real responses for all 5)
    similarity_scores = judge_held_out_similarity(
        client_anthropic, config,
        simulated_response=simulated,
        real_response=scenario['real_response'],
        context=scenario['context'],
    )

    result = {
        'scenario': scenario_name,
        'simulated': simulated,
        **criteria_scores,
        **similarity_scores,
    }
    generation_results.append(result)

    print(f'\nReal response (excerpt):\n  {scenario["real_response"]}')
    print(f'\nSimulated response (excerpt):\n  {simulated}')
    print(f'\nBehavioral Plausibility:     {criteria_scores.get("behavioral_plausibility")}/5')
    print(f'Persona Consistency:         {criteria_scores.get("persona_consistency")}/5')
    print(f'Intervention Responsiveness: {criteria_scores.get("intervention_responsiveness")}/5')
    print(f'Held-out Similarity:         {similarity_scores.get("held_out_similarity")}/10')
    print(f'Criteria critique:   {criteria_scores.get("critique")}')
    print(f'Similarity critique: {similarity_scores.get("critique")}')

[memory] Loading 55 seed memories for Margaret Chen...
  [1/55] 'I had two Fire Marshals from the Berkeley fire department co...' (importance=7)
  [2/55] 'The fire marshals told me, surprisingly, that even in the mo...' (importance=6)
  [3/55] 'We had to make sure there's enough space between the tree le...' (importance=5)
  [4/55] 'We had all our trees trimmed to create proper spacing from t...' (importance=6)
  [5/55] 'We have two Japanese maple trees that are coming through dec...' (importance=7)
  [6/55] 'I've already taken out a bunch of bushes that are within fiv...' (importance=6)
  [7/55] 'There are volunteer heads of districts up here, like these t...' (importance=4)
  [8/55] 'We had a guy who was actually an engineer who was the head o...' (importance=5)
  [9/55] 'We have a deadline of June 1st to get the first level of stu...' (importance=7)
  [10/55] 'On the side of our house, we have a very large silk tassel, ...' (importance=5)
  [11/55] 'The fire guys said they're not su

In [26]:
print(f'\n{"="*70}')
print('GENERATION EVALUATION — PER-SCENARIO RESULTS')
print(f'{"="*70}')
print(f'\n{"Scenario":<30} {"Plaus":>6} {"Persona":>8} {"Resp":>5} {"HO Sim":>7}')
print('-' * 60)
for r in generation_results:
    print(
        f'{r["scenario"]:<30} '
        f'{str(r.get("behavioral_plausibility")):>5}/5 '
        f'{str(r.get("persona_consistency")):>7}/5 '
        f'{str(r.get("intervention_responsiveness")):>4}/5 '
        f'{str(r.get("held_out_similarity")):>6}/10'
    )


def safe_avg(values):
    vals = [v for v in values if v is not None]
    return round(sum(vals) / len(vals), 2) if vals else None


avg_plausibility = safe_avg([r.get('behavioral_plausibility')     for r in generation_results])
avg_persona      = safe_avg([r.get('persona_consistency')         for r in generation_results])
avg_responsive   = safe_avg([r.get('intervention_responsiveness') for r in generation_results])
avg_similarity   = safe_avg([r.get('held_out_similarity')         for r in generation_results])

n = len(generation_results)
print(f'\n{"="*70}')
print(f'AGGREGATE SCORES FOR {AGENT_NAME.upper()}  (n={n} scenarios)')
print(f'{"="*70}')
print(f'  1. Behavioral Plausibility:     {avg_plausibility} / 5')
print(f'  2. Persona Consistency:         {avg_persona} / 5')
print(f'  3. Intervention Responsiveness: {avg_responsive} / 5')
print(f'  4. Held-out Similarity:         {avg_similarity} / 10  (Beth has real responses for all {n} scenarios)')


GENERATION EVALUATION — PER-SCENARIO RESULTS

Scenario                        Plaus  Persona  Resp  HO Sim
------------------------------------------------------------
insurance_non_renewal              5/5       4/5    5/5      8/10
defensible_space_inspection        5/5       4/5    4/5      4/10
neighbor_pressure                  4/5       5/5    4/5      9/10
firewise_outreach                  5/5       5/5    4/5      6/10
resources_assistance               4/5       4/5    4/5      5/10

AGGREGATE SCORES FOR MARGARET CHEN  (n=5 scenarios)
  1. Behavioral Plausibility:     4.6 / 5
  2. Persona Consistency:         4.4 / 5
  3. Intervention Responsiveness: 4.2 / 5
  4. Held-out Similarity:         6.4 / 10  (Beth has real responses for all 5 scenarios)
